# **Experiment 11: LoRA / Adapter-Based Parameter-Efficient Fine-Tuning**

Aim:
To implement parameter-efficient fine-tuning using LoRA (Low-Rank Adaptation) and reduce the number of trainable parameters.

Brief Theory:
LoRA freezes the original pre-trained model and adds small trainable low-rank matrices to selected layers. Instead of updating all model parameters, only these additional parameters are trained. Therefore, LoRA requires significantly fewer trainable parameters and less memory.

In [5]:
# Experiment 11
# LoRA / Adapter-Based Parameter-Efficient Fine-Tuning

!pip install -q transformers peft datasets accelerate sentencepiece

import torch
import json
import time

print("Student Name: Sayem")
print("Student ID: 25/SET/MTCE/014")
print("Libraries installed successfully!")

Student Name: Sayem
Student ID: 25/SET/MTCE/014
Libraries installed successfully!


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded successfully!")
print("Model:", model_name)
print("Vocabulary Size:", tokenizer.vocab_size)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded successfully!
Model: distilgpt2
Vocabulary Size: 50257


In [7]:
# Check model modules related to attention

for name, module in model.named_modules():
    if "c_attn" in name:
        print("LoRA Target Module:", name)

LoRA Target Module: transformer.h.0.attn.c_attn
LoRA Target Module: transformer.h.1.attn.c_attn
LoRA Target Module: transformer.h.2.attn.c_attn
LoRA Target Module: transformer.h.3.attn.c_attn
LoRA Target Module: transformer.h.4.attn.c_attn
LoRA Target Module: transformer.h.5.attn.c_attn


In [9]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none"
)

print("LoRA configuration created successfully!")
print("Rank (r):", 8)
print("Alpha:", 16)
print("Dropout:", 0.1)
print("Target Module: c_attn")

LoRA configuration created successfully!
Rank (r): 8
Alpha: 16
Dropout: 0.1
Target Module: c_attn


In [12]:
!pip uninstall -y torchao
!pip install -q -U peft transformers accelerate
print("Conflict package removed successfully!")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.9 MB/s eta 0:00:00
Conflict package removed successfully!


In [13]:
from peft import get_peft_model

peft_model = get_peft_model(
    model,
    lora_config
)

print("LoRA adapter added successfully!")

LoRA adapter added successfully!


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  


In [14]:
peft_model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


In [15]:
trainable_params = 0
frozen_params = 0

for name, param in peft_model.named_parameters():

    if param.requires_grad:
        trainable_params += param.numel()
    else:
        frozen_params += param.numel()

print("========== Parameter Status ==========")
print("Trainable Parameters:", trainable_params)
print("Frozen Parameters:", frozen_params)
print("Total Parameters:", trainable_params + frozen_params)

trainable_percentage = (
    trainable_params /
    (trainable_params + frozen_params)
) * 100

print("Trainable Percentage:", round(trainable_percentage, 4), "%")

========== Parameter Status ==========
Trainable Parameters: 147456
Frozen Parameters: 81912576
Total Parameters: 82060032
Trainable Percentage: 0.1797 %


In [16]:
data = [
    {
        "instruction": "What is Artificial Intelligence?",
        "response": "Artificial Intelligence enables machines to perform tasks that normally require human intelligence."
    },
    {
        "instruction": "What is Machine Learning?",
        "response": "Machine Learning allows computers to learn patterns from data."
    },
    {
        "instruction": "What is Deep Learning?",
        "response": "Deep Learning uses multi-layer neural networks to learn complex patterns."
    },
    {
        "instruction": "What is Computer Vision?",
        "response": "Computer Vision enables computers to understand images and videos."
    },
    {
        "instruction": "What is Generative AI?",
        "response": "Generative AI can create new text, images, audio and other content."
    },
    {
        "instruction": "What is Fine-Tuning?",
        "response": "Fine-Tuning adapts a pre-trained model to a specific task."
    }
]

print("Custom dataset created!")
print("Total samples:", len(data))

Custom dataset created!
Total samples: 6


In [17]:
from datasets import Dataset

dataset = Dataset.from_list(data)

def tokenize_function(examples):

    texts = [
        "Instruction: " + instruction +
        "\nResponse: " + response
        for instruction, response
        in zip(
            examples["instruction"],
            examples["response"]
        )
    ]

    return tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

print("Dataset tokenized successfully!")

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset tokenized successfully!


In [18]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator configured!")
print("Training objective: Causal Language Modeling")

Data collator configured!
Training objective: Causal Language Modeling


In [19]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora_distilgpt2",
    num_train_epochs=1,
    max_steps=5,
    per_device_train_batch_size=2,
    learning_rate=1e-4,
    logging_steps=1,
    save_steps=5,
    save_total_limit=1,
    report_to="none"
)

print("========== LoRA Training Configuration ==========")
print("Epochs:", 1)
print("Max Steps:", 5)
print("Batch Size:", 2)
print("Learning Rate:", 1e-4)
print("Sequence Length:", 128)

========== LoRA Training Configuration ==========
Epochs: 1
Max Steps: 5
Batch Size: 2
Learning Rate: 0.0001
Sequence Length: 128


In [20]:
from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer configured successfully!")

Trainer configured successfully!


In [21]:
print("Starting LoRA fine-tuning...")

start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time

print("\n========== LoRA Training Completed ==========")
print("Training Steps:", train_result.global_step)
print("Training Time:", round(training_time, 2), "seconds")

Starting LoRA fine-tuning...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.890049
2,3.821734
3,3.994787
4,3.700242
5,3.859600



========== LoRA Training Completed ==========
Training Steps: 5
Training Time: 13.43 seconds


In [22]:
adapter_path = "./lora_distilgpt2_adapter"

peft_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully!")
print("Adapter Path:", adapter_path)

LoRA adapter saved successfully!
Adapter Path: ./lora_distilgpt2_adapter


In [23]:
prompt = "Instruction: What is Machine Learning?\nResponse:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

peft_model.eval()

with torch.no_grad():

    output = peft_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

result = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("========== LoRA Inference ==========")
print("Input:", prompt)
print("Output:", result)

========== LoRA Inference ==========
Input: Instruction: What is Machine Learning?
Response:
Output: Instruction: What is Machine Learning?
Response: Machine Learning is a new way to learn and learn. Machine learning is a new way to learn and learn. Machine learning is a new way to learn and learn. Machine learning is a new way to


In [24]:
import os

print("========== Saved Adapter Files ==========")

for file in os.listdir(adapter_path):
    print(file)

========== Saved Adapter Files ==========
adapter_config.json
tokenizer.json
tokenizer_config.json
README.md
adapter_model.safetensors


In [25]:
total_params = trainable_params + frozen_params

print("\n========== LoRA Parameter Efficiency ==========")

print("Total Parameters:", total_params)
print("Trainable Parameters:", trainable_params)
print("Frozen Parameters:", frozen_params)
print("Trainable Percentage:", round(trainable_percentage, 4), "%")
print("Parameter Reduction:",
      round(100 - trainable_percentage, 4), "%")


========== LoRA Parameter Efficiency ==========
Total Parameters: 82060032
Trainable Parameters: 147456
Frozen Parameters: 81912576
Trainable Percentage: 0.1797 %
Parameter Reduction: 99.8203 %


**Observation / Result**

**Observation:**
LoRA was applied to the pre-trained DistilGPT-2 model using the PEFT library. The original model parameters were frozen while a small number of low-rank adapter parameters were made trainable.
A short LoRA fine-tuning run was successfully performed and the adapter weights were saved separately.

**Result:** The experiment demonstrated that LoRA can adapt a pre-trained language model while training only a small fraction of the total model parameters. The saved adapter was successfully loaded into the model and used for inference.